In [1]:
import os
import requests
import pandas as pd  # <-- ADD THIS LINE

# Read the key from an environment variable instead of hardcoding it
API_KEY = os.environ.get("OPENAQ_API_KEY", "bf008b449b0290c59067e4c0072c9dacb6d6fe10c63ebf174d3a9736c2c334a8")
BASE_URL = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}

# Kathmandu Valley bounding box: min_lon, min_lat, max_lon, max_lat
KATHMANDU_BBOX = "85.20,27.55,85.60,27.85"
PM25_PARAMETER_ID = 2

def get_valley_stations():
    """Return all stations inside the Kathmandu Valley that measure PM2.5."""
    params = {
        "bbox": KATHMANDU_BBOX,
        "parameters_id": PM25_PARAMETER_ID,
        "limit": 100,
    }
    response = requests.get(f"{BASE_URL}/locations", headers=HEADERS, params=params, timeout=30)
    response.raise_for_status()
    return response.json().get("results", [])

def get_latest_pm25(location):
    """Return (value, timestamp) of the latest PM2.5 reading for a location."""
    pm25_sensor_ids = {
        sensor["id"]
        for sensor in location.get("sensors", [])
        if sensor.get("parameter", {}).get("id") == PM25_PARAMETER_ID
    }

    response = requests.get(
        f"{BASE_URL}/locations/{location['id']}/latest",
        headers=HEADERS,
        timeout=30,
    )
    response.raise_for_status()

    for reading in response.json().get("results", []):
        if reading.get("sensorsId") in pm25_sensor_ids:
            timestamp = reading.get("datetime", {}).get("local")
            return reading.get("value"), timestamp
    return None, None

def main():
    print("Fetching Kathmandu Valley PM2.5 stations from OpenAQ v3...\n")
    try:
        stations = get_valley_stations()
    except requests.HTTPError as err:
        print(f"Request failed: {err}\n{err.response.text}")
        return

    if not stations:
        print("No stations found in the specified area.")
        return

    print(f"Stations found: {len(stations)}\n")
    
    # --- NEW: Create a list to hold our structured data ---
    clean_data = []

    for station in stations:
        try:
            value, timestamp = get_latest_pm25(station)
        except requests.HTTPError as err:
            value, timestamp = None, f"error {err.response.status_code}"

        coords = station.get("coordinates", {})
        
        # --- NEW: Append a clean dictionary for each station ---
        clean_data.append({
            "station_id": station.get("id"),
            "station_name": station.get("name"),
            "latitude": coords.get("latitude"),
            "longitude": coords.get("longitude"),
            "pm25_value": value,
            "last_updated": timestamp
        })

    # --- NEW: Convert to Pandas DataFrame ---
    df = pd.DataFrame(clean_data)
    
    print("--- DataFrame Preview ---")
    print(df.head())
    
    print("\n--- Data Info (Checking for missing values) ---")
    print(df.info())

    # --- NEW: Save to CSV ---
    # We use index=False so Pandas doesn't add an unnecessary row-number column
    output_path = "data/raw/kathmandu_pm25_raw.csv"
    df.to_csv(output_path, index=False)
    print(f"\n✅ Success! Data saved to: {output_path}")

if __name__ == "__main__":
    main()

Fetching Kathmandu Valley PM2.5 stations from OpenAQ v3...

Stations found: 62

--- DataFrame Preview ---
   station_id                                    station_name   latitude  \
0        3459                               Embassy Kathmandu  27.738703   
1        3460                            Phora Durbar Kathman  27.712464   
2     1236017                           Dhathutole, Handigaun  27.727502   
3     2848864                               Dabali, Handigaun  27.681891   
4     4121681  Kathmandu Institute of Applied Sciences (KIAS)  27.671693   

   longitude  pm25_value               last_updated  
0  85.336206        21.0  2026-06-18T05:00:00+05:45  
1  85.315703        71.0  2025-03-24T20:00:00+05:45  
2  85.330135        47.7  2026-06-01T09:45:00+05:45  
3  85.287078        42.5  2026-10-19T08:45:00+05:45  
4  85.301670        28.3  2026-11-21T16:45:00+05:45  

--- Data Info (Checking for missing values) ---
<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data 